# Image Classification with Transformers
### Author: Cole Drumheller

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 1337
torch.manual_seed(seed)

img_size = 32            # CIFAR-10 images are 32x32
patch_size = 4           # 4x4 patches -> (32/4)^2 = 64 tokens
in_chans = 3
num_classes = 10

# model capacity
embed_dim = 192
num_heads = 6
mlp_ratio = 4
num_layers = 6
dropout = 0.1

# training
batch_size = 128
epochs = 20
lr = 3e-4
weight_decay = 0.05

# misc
save_dir = "./vit_cifar"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# ---------------------------
# Data: CIFAR-10
# ---------------------------
transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

train_ds = torchvision.datasets.CIFAR10(root=".", train=True, download=True, transform=transform_train)
test_ds = torchvision.datasets.CIFAR10(root=".", train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ---------------------------
# ViT components
# ---------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, embed_dim):
        super().__init__()
        assert img_size % patch_size == 0, "Image size must be divisible by patch size"
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_size = patch_size
        # Use a conv layer to produce patch embeddings (efficient)
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.proj(x)                # (B, embed_dim, H/ps, W/ps)
        x = x.flatten(2)                # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)           # (B, num_patches, embed_dim)
        return x

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, in_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

In [ ]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, dropout=0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x)                      # (B, N, 3*C)
        qkv = qkv.reshape(B, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)       # (3, B, heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        out = (attn @ v)                       # (B, heads, N, head_dim)
        out = out.transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        out = self.proj_drop(out)
        return out

In [ ]:
class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads=num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(dim, hidden_dim, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self,
                 img_size=32,
                 patch_size=4,
                 in_chans=3,
                 num_classes=10,
                 embed_dim=192,
                 depth=6,
                 num_heads=6,
                 mlp_ratio=4.0,
                 dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches

        # class token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        # positional embedding
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, embed_dim))
        self.pos_drop = nn.Dropout(p=dropout)

        # transformer encoder
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, dropout=dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # classification head
        self.head = nn.Linear(embed_dim, num_classes)

        # init
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

        def _init_weights(self, m):
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.zeros_(m.bias)
                nn.init.ones_(m.weight)

        def forward(self, x):
            B = x.shape[0]
            x = self.patch_embed(x)             # (B, num_patches, embed_dim)
            cls_tokens = self.cls_token.expand(B, -1, -1)  # (B,1,embed_dim)
            x = torch.cat((cls_tokens, x), dim=1)           # (B, 1+num_patches, embed_dim)
            x = x + self.pos_embed
            x = self.pos_drop(x)

            for blk in self.blocks:
                x = blk(x)

            x = self.norm(x)
            cls = x[:, 0]                       # take cls token
            out = self.head(cls)
            return out

In [ ]:
# ---------------------------
# Instantiate model, loss, optimizer
# ---------------------------
model = VisionTransformer(
    img_size=img_size,
    patch_size=patch_size,
    in_chans=in_chans,
    num_classes=num_classes,
    embed_dim=embed_dim,
    depth=num_layers,
    num_heads=num_heads,
    mlp_ratio=mlp_ratio,
    dropout=dropout
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))

In [ ]:
# ---------------------------
# Training & evaluation helpers
# ---------------------------
def train_one_epoch(epoch):
    model.train()
    pbar = tqdm(train_loader, desc=f"Train {epoch}")
    total_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in pbar:
        imgs = imgs.to(device)
        labels = labels.to(device)

        logits = model(imgs)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=total_loss / total, acc=correct/total)

    return total_loss / total, correct / total

def evaluate():
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            total_loss += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

In [ ]:
# # ---------------------------
# # Main loop
# # ---------------------------
# train_losses = []
# train_accs = []
# val_losses = []
# val_accs = []

# best_val_acc = 0.0
# for epoch in range(1, epochs + 1):
#     train_loss, train_acc = train_one_epoch(epoch)
#     val_loss, val_acc = evaluate()

#     train_losses.append(train_loss)
#     train_accs.append(train_acc)
#     val_losses.append(val_loss)
#     val_accs.append(val_acc)

#     print(f"Epoch {epoch}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

#     # save best
#     if val_acc > best_val_acc:
#         best_val_acc = val_acc
#         torch.save({
#             "model_state": model.state_dict(),
#             "optimizer_state": optimizer.state_dict(),
#             "epoch": epoch
#         }, os.path.join(save_dir, "best_vit.pth"))
#         print(f"Saved best model (val_acc={best_val_acc:.4f})")

In [ ]:
# # ---------------------------
# # Plot metrics
# # ---------------------------
# plt.figure(figsize=(10,4))
# plt.subplot(1,2,1)
# plt.plot(train_losses, label="train_loss")
# plt.plot(val_losses, label="val_loss")
# plt.legend()
# plt.title("Loss")

# plt.subplot(1,2,2)
# plt.plot(train_accs, label="train_acc")
# plt.plot(val_accs, label="val_acc")
# plt.legend()
# plt.title("Accuracy")
# plt.show()

# print("Best validation accuracy:", best_val_acc)